In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install torch==2.3.0 torchtext==0.18.0 torchaudio==2.3.0 torchvision==0.18.0 transformers==4.50.2 evaluate==0.4.3 datasets==3.5.0

In [ ]:
import os
import numpy as np
import pandas as pd
from torch import Tensor
import torch
import torch.nn as nn
from torch.nn import Transformer
import math
from torch.nn.utils.rnn import pad_sequence
import random
from tqdm import tqdm
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchtext.transforms import BERTTokenizer
from torch.optim import AdamW
import sys
from timeit import default_timer as timer
%matplotlib inline
from torchtext.vocab import build_vocab_from_iterator
from torchtext.data.utils import get_tokenizer
from typing import Iterable, List
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

In [ ]:
!pip install portalocker

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
df = pd.read_csv('train.csv')

validation_data = pd.read_csv('val.csv')
test_data = pd.read_csv('test.csv')

df = df.drop_duplicates()
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna()


validation_data = validation_data.drop_duplicates()
validation_data = validation_data.replace([np.inf, -np.inf], np.nan)
validation_data = validation_data.dropna()

test_data = test_data.drop_duplicates()
test_data = test_data.replace([np.inf, -np.inf], np.nan)
test_data = test_data.dropna()


In [ ]:
len(df), len(validation_data), len(test_data)

In [ ]:

train_df = df
valid_df = validation_data
test_df = test_data

In [ ]:
len(train_df), len(valid_df), len(test_df)

In [ ]:
train_source_sentences = list(train_df['source sentence'].values)
train_target_sentences = list(train_df['target sentence'].values)

valid_source_sentences = list(valid_df['source sentence'].values)
valid_target_sentences = list(valid_df['target sentence'].values)

test_source_sentences = list(test_df['source sentence'].values)
test_target_sentences = list(test_df['target sentence'].values)

print(f"Train Source: {len(train_source_sentences)}")
print(f"Test Source: {len(test_source_sentences)}")
print(f"Train Target: {len(train_target_sentences)}")
print(f"Test Target: {len(test_target_sentences)}")
print(f"Valid Source: {len(valid_source_sentences)}")
print(f"Valid Target: {len(valid_target_sentences)}")

In [ ]:
max(len(x) for x in train_source_sentences), max(len(x) for x in train_target_sentences), max(len(x) for x in test_source_sentences), max(len(x) for x in test_target_sentences),

In [ ]:
PERCENTILE = 97
print( f"{PERCENTILE}th percentile length in Train Source: {np.percentile([len(x) for x in train_source_sentences], PERCENTILE)}" )
print( f"{PERCENTILE}th percentile length in Train Target: {np.percentile([len(x) for x in train_target_sentences], PERCENTILE)}" )

print( f"{PERCENTILE}th percentile length in Test Source: {np.percentile([len(x) for x in test_source_sentences], PERCENTILE)}" )
print( f"{PERCENTILE}th percentile length in Test Target: {np.percentile([len(x) for x in test_target_sentences], PERCENTILE)}" )

In [ ]:
class TextDataset(Dataset):

    def __init__(self, source_sentences, target_sentences):
        self.source_sentences = source_sentences
        self.target_sentences = target_sentences

    def __len__(self):
        return len(self.source_sentences)

    def __getitem__(self, idx):
        return self.source_sentences[idx], self.target_sentences[idx]

In [ ]:
train_dataset = TextDataset(train_source_sentences, train_target_sentences)
val_dataset = TextDataset(valid_source_sentences, valid_target_sentences)
eval_dataset = TextDataset(test_source_sentences, test_target_sentences)

In [ ]:
train_dataset.__len__() , val_dataset.__len__(), eval_dataset.__len__()

In [ ]:
train_dataset.__getitem__(0), val_dataset.__getitem__(0), eval_dataset.__getitem__(0)

In [ ]:
VOCAB_FILE = "https://huggingface.co/bert-base-uncased/resolve/main/vocab.txt"
bert_tokenizer = BERTTokenizer(vocab_path=VOCAB_FILE, do_lower_case=True, return_tokens=True)

SRC_LANGUAGE = 'src_en'
TGT_LANGUAGE = 'tgt_en'

token_transform = {}
vocab_transform = {}

In [ ]:
token_transform[SRC_LANGUAGE] = bert_tokenizer
token_transform[TGT_LANGUAGE] = bert_tokenizer


# helper function to yield list of tokens
def yield_tokens(data_iter: Iterable, language: str) -> List[str]:
    language_index = {SRC_LANGUAGE: 0, TGT_LANGUAGE: 1}

    for data_sample in tqdm(data_iter):
        yield token_transform[language](data_sample[language_index[language]])

# Define special symbols and indices
UNK_IDX, PAD_IDX, BOS_IDX, EOS_IDX = 0, 1, 2, 3
# Make sure the tokens are in order of their indices to properly insert them in vocab
special_symbols = ['<unk>', '<pad>', '<bos>', '<eos>']

for ln in [SRC_LANGUAGE, TGT_LANGUAGE]:
    # Training data Iterator
    # Create torchtext's Vocab object
    vocab_transform[ln] = build_vocab_from_iterator(yield_tokens(train_dataset, ln),
                                                    min_freq=1,
                                                    specials=special_symbols,
                                                    special_first=True)

# Set ``UNK_IDX`` as the default index. This index is returned when the token is not found.
# If not set, it throws ``RuntimeError`` when the queried token is not found in the Vocabulary.
for ln in [SRC_LANGUAGE, TGT_LANGUAGE]:
    vocab_transform[ln].set_default_index(UNK_IDX)

In [ ]:
len(vocab_transform['tgt_en']), len(vocab_transform['src_en'])

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Device: {DEVICE}")

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
from typing import Tuple, Optional

class PositionalEncoding(nn.Module):
    def __init__(self, emb_size: int, dropout: float, maxlen: int = 1000):
        super(PositionalEncoding, self).__init__()
        den = torch.exp(- torch.arange(0, emb_size, 2) * math.log(10000) / emb_size)
        pos = torch.arange(0, maxlen).reshape(maxlen, 1)
        pos_embedding = torch.zeros((maxlen, emb_size))
        pos_embedding[:, 0::2] = torch.sin(pos * den)
        pos_embedding[:, 1::2] = torch.cos(pos * den)
        pos_embedding = pos_embedding.unsqueeze(-2)

        self.dropout = nn.Dropout(dropout)
        self.register_buffer('pos_embedding', pos_embedding)

    def forward(self, token_embedding: Tensor):
        return self.dropout(token_embedding + self.pos_embedding[:token_embedding.size(0), :])


class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size: int, emb_size):
        super(TokenEmbedding, self).__init__()
        self.embedding = nn.Embedding(vocab_size, emb_size)
        self.emb_size = emb_size

    def forward(self, tokens: Tensor):
        return self.embedding(tokens.long()) * math.sqrt(self.emb_size)

    
class RMSNorm(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-6):
        super(RMSNorm, self).__init__()
        self.d_model = d_model
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(d_model))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        rms = torch.sqrt(torch.mean(x ** 2, dim=-1, keepdim=True) + self.eps)
        return self.scale * (x / rms)


class Seq2SeqTransformer(nn.Module):
    def __init__(self,
                 num_encoder_layers: int,
                 num_decoder_layers: int,
                 emb_size: int,
                 nhead: int,
                 src_vocab_size: int,
                 tgt_vocab_size: int,
                 dim_feedforward: int,
                 dropout: float = 0.1,
                 reduction_ratio: float = 0.2):
        """
        Seq2Seq Transformer with learnable importance-based token selection.
        
        Uses a simple and effective approach:
        1. Importance head learns to score token salience
        2. Top-k selection picks the most important tokens
        
        Args:
            reduction_ratio: Fraction of tokens to keep (0 < r ≤ 1)
        """
        super(Seq2SeqTransformer, self).__init__()
        self.reduction_ratio = reduction_ratio
        
        self.transformer = nn.Transformer(
            d_model=emb_size,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout
        )
        
        self.generator = nn.Linear(emb_size, tgt_vocab_size)
        self.src_tok_emb = TokenEmbedding(src_vocab_size, emb_size)
        self.tgt_tok_emb = TokenEmbedding(tgt_vocab_size, emb_size)
        self.positional_encoding = PositionalEncoding(emb_size, dropout=dropout)
        self.rms_norm = RMSNorm(emb_size)
        
        # Learnable importance head - maps embeddings to importance scores
        self.importance_head = nn.Sequential(
            nn.Linear(emb_size, emb_size // 4),  # Bottleneck for efficiency
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(emb_size // 4, 1)  # Output logits (will apply sigmoid later)
        )
        
        # Temperature scaling parameter for importance scores
        self.learned_temp_scale = nn.Parameter(torch.tensor(0.5))

    def forward(self,
                src: Tensor,
                tgt: Tensor,
                src_mask: Tensor,
                tgt_mask: Tensor,
                src_padding_mask: Tensor,
                tgt_padding_mask: Tensor,
                memory_key_padding_mask: Tensor):
        
        src_emb = self.positional_encoding(self.src_tok_emb(src))
        tgt_emb = self.positional_encoding(self.tgt_tok_emb(tgt))

        # Encode
        encoder_out = self.transformer.encoder(src_emb, src_mask, src_padding_mask)
        
        # Select tokens with importance head
        selected_tokens, selected_padding_mask = self.select_tokens(
            encoder_out, src, src_padding_mask
        )

        # print(f"encoder tokens: {encoder_out}\nencoder padding mask: {src_padding_mask}\n")
        # print(f"\nselected tokens: {selected_tokens}\nselected padding mask: {selected_padding_mask}\n")

        # Decode with selected tokens
        decoder_out = self.transformer.decoder(
            tgt_emb, selected_tokens, tgt_mask, None,
            tgt_padding_mask, selected_padding_mask
        )

        return self.generator(decoder_out)

    def select_tokens(
        self, 
        encoder_out: Tensor, 
        src: Tensor, 
        src_padding_mask: Tensor
    ) -> Tuple[Tensor, Tensor]:
        """
        Select tokens using the importance head with top-k selection.
        """
        # No reduction case
        if self.reduction_ratio == 1.0:
            return encoder_out, src_padding_mask

        # Normalize encoder output
        encoder_out_rms = self.rms_norm(encoder_out)
        max_len = encoder_out.size(0)

        selected_tokens = []
        selected_padding_masks = []

        for i in range(encoder_out.size(1)):  # Iterate over batch dimension
            # Find sentence boundaries (token_id 2=BOS, 3=EOS)
            start_indices = torch.where(src[:, i] == 2)[0]
            end_indices = torch.where(src[:, i] == 3)[0]

            sentence_padding_mask = torch.full((max_len,), True, dtype=torch.bool, device=encoder_out.device)

            for start, end in zip(start_indices, end_indices):
                token_length = end - start - 1  # Exclude BOS and EOS
                
                # Dynamically adjust temperature based on length
                temperature = F.softplus(self.learned_temp_scale) * (1 + 1 / (token_length + 1))

                # Extract sentence span for importance scoring
                sentence_span = encoder_out_rms[start:end+1, i, :]
                
                # Compute importance scores using a learnable importance head
                # Shape: [sentence_length, emb_dim] -> [sentence_length, 1]
                importance_logits = self.importance_head(sentence_span)
                
                # Extract middle token logits BEFORE softmax (exclude BOS and EOS)
                middle_logits = importance_logits[1:-1].squeeze(-1)  # Shape: [token_length]
                
                # CRITICAL FIX: Mask out padding tokens before softmax
                # Get padding mask for middle tokens (exclude BOS and EOS)
                middle_padding_mask = src_padding_mask[i, start+1:end]  # Shape: [token_length]
                # print(f"middle logits: {middle_logits}\nmiddle padding mask: {middle_padding_mask}\n")
                
                # Set padding token logits to -inf so they get 0 probability after softmax
                middle_logits = middle_logits.masked_fill(middle_padding_mask, float('-inf'))
                # print(f"middle logits v2: {middle_logits}\nmiddle padding mask v2: {middle_padding_mask}\n")
                
                # Apply softmax with temperature scaling only on middle tokens
                # Padding tokens will now have 0 importance scores
                importance_scores = F.softmax(middle_logits / temperature, dim=0)

                # Always include start token
                sentence_tokens = [encoder_out[start, i, :].unsqueeze(0)]
                padding_mask = [src_padding_mask[i, start].unsqueeze(0)]

                if math.ceil(token_length * self.reduction_ratio) <= 3:
                    # Keep all middle tokens if reduction is too small
                    selected_span = encoder_out[start + 1:end, i, :]
                    selected_mask = src_padding_mask[i, start + 1:end]
                else:
                    # Calculate number of tokens to keep
                    reduced_length = max(4, int(math.ceil(token_length * self.reduction_ratio)))
                    
                    # Select top-k tokens based on importance scores
                    # This will automatically exclude padding tokens as they have 0 importance
                    _, top_indices = torch.topk(importance_scores, reduced_length, dim=0)
                    
                    # Convert to global indices
                    global_top_indices = start + 1 + top_indices
                    global_top_indices, _ = torch.sort(global_top_indices)
                    
                    selected_span = encoder_out[global_top_indices, i, :]
                    selected_mask = src_padding_mask[i, global_top_indices]

                # Append selected middle tokens
                sentence_tokens.extend([selected_span[j].unsqueeze(0) for j in range(selected_span.size(0))])
                padding_mask.extend([selected_mask[j].unsqueeze(0) for j in range(selected_mask.size(0))])

                # Append end token
                sentence_tokens.append(encoder_out[end, i, :].unsqueeze(0))
                padding_mask.append(src_padding_mask[i, end].unsqueeze(0))

                # Concatenate tokens and pad to max length
                sentence_tokens = torch.cat(sentence_tokens, dim=0)
                padded_sentence_tokens = F.pad(sentence_tokens, (0, 0, 0, max_len - sentence_tokens.size(0)))
                selected_tokens.append(padded_sentence_tokens)

                # Update padding mask
                sentence_padding_mask[:len(padding_mask)] = torch.cat(padding_mask)
                sentence_padding_mask[len(padding_mask):] = True
                selected_padding_masks.append(sentence_padding_mask.clone())

        # Stack all selected tokens and masks
        selected_tokens = torch.stack(selected_tokens, dim=1)
        selected_padding_mask = torch.stack(selected_padding_masks, dim=0)

        # print(f"selected tokens: {selected_tokens}\nselected padding mask: {selected_padding_mask}\n")

        return selected_tokens, selected_padding_mask

    def encode(self, src: Tensor, src_mask: Tensor):
        return self.transformer.encoder(
            self.positional_encoding(self.src_tok_emb(src)), 
            src_mask
        )

    def decode(self, tgt: Tensor, memory: Tensor, tgt_mask: Tensor):
        return self.transformer.decoder(
            self.positional_encoding(self.tgt_tok_emb(tgt)), 
            memory, 
            tgt_mask
        )

In [ ]:
def generate_square_subsequent_mask(sz):
    mask = (torch.triu(torch.ones((sz, sz), device=DEVICE)) == 1).transpose(0, 1)
    mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
    return mask


def create_mask(src, tgt):
    src_seq_len = src.shape[0]
    tgt_seq_len = tgt.shape[0]

    tgt_mask = generate_square_subsequent_mask(tgt_seq_len)
    src_mask = torch.zeros((src_seq_len, src_seq_len),device=DEVICE).type(torch.bool)

    src_padding_mask = (src == PAD_IDX).transpose(0, 1)
    tgt_padding_mask = (tgt == PAD_IDX).transpose(0, 1)
    return src_mask, tgt_mask, src_padding_mask, tgt_padding_mask

In [ ]:
torch.manual_seed(0)

SRC_VOCAB_SIZE = len(vocab_transform[SRC_LANGUAGE])
TGT_VOCAB_SIZE = len(vocab_transform[TGT_LANGUAGE])
EMB_SIZE = 512
NHEAD = 8
FFN_HID_DIM = 2048
NUM_ENCODER_LAYERS = 6
NUM_DECODER_LAYERS = 6

model = Seq2SeqTransformer(NUM_ENCODER_LAYERS, NUM_DECODER_LAYERS, EMB_SIZE,
                                 NHEAD, SRC_VOCAB_SIZE, TGT_VOCAB_SIZE, FFN_HID_DIM)

for p in model.parameters():
    if p.dim() > 1:
        nn.init.xavier_uniform_(p)

model = model.to(DEVICE)

loss_fn = torch.nn.CrossEntropyLoss(ignore_index=PAD_IDX, label_smoothing=0.1)

In [ ]:
# helper function to club together sequential operations
def sequential_transforms(*transforms):
    def func(txt_input):
        for transform in transforms:
            txt_input = transform(txt_input)
        return txt_input
    return func

# function to add BOS/EOS and create tensor for input sequence indices
def tensor_transform(token_ids: List[int]):
    return torch.cat((torch.tensor([BOS_IDX]),
                      torch.tensor(token_ids),
                      torch.tensor([EOS_IDX])))

# ``src`` and ``tgt`` language text transforms to convert raw strings into tensors indices
text_transform = {}
for ln in [SRC_LANGUAGE, TGT_LANGUAGE]:
    text_transform[ln] = sequential_transforms(token_transform[ln], #Tokenization
                                               vocab_transform[ln], #Numericalization
                                               tensor_transform) # Add BOS/EOS and create tensor


# function to collate data samples into batch tensors
def collate_fn(batch):
    src_batch, tgt_batch = [], []
    for src_sample, tgt_sample in batch:
        src_batch.append(text_transform[SRC_LANGUAGE](src_sample.rstrip("\n")))
        tgt_batch.append(text_transform[TGT_LANGUAGE](tgt_sample.rstrip("\n")))

    src_batch = pad_sequence(src_batch, padding_value=PAD_IDX)
    tgt_batch = pad_sequence(tgt_batch, padding_value=PAD_IDX)
    return src_batch, tgt_batch

In [ ]:
train_dataloader = DataLoader(train_dataset, batch_size=32, collate_fn=collate_fn, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=32, collate_fn=collate_fn, shuffle=False)

In [ ]:
def evaluate(model):
    model.eval()
    losses = 0

    for src, tgt in val_dataloader:
        src = src.to(DEVICE)
        tgt = tgt.to(DEVICE)

        tgt_input = tgt[:-1, :]

        src_mask, tgt_mask, src_padding_mask, tgt_padding_mask = create_mask(src, tgt_input)

        logits = model(src, tgt_input, src_mask, tgt_mask,src_padding_mask, tgt_padding_mask, src_padding_mask)

        tgt_out = tgt[1:, :]
        loss = loss_fn(logits.reshape(-1, logits.shape[-1]), tgt_out.reshape(-1))
        losses += loss.item()

    return losses / len(list(val_dataloader))

# **Hyper Parameter**

In [ ]:
# 11 epoch - 1st run
# 21 epoch - 2nd run
## 33 epoch - 3rd run

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
N_EPOCHS = 32
epoch = 0
loss = 10e9

PATH = '/kaggle/working/TextEconomizer_WMT19_20_V3.pth'
BEST_PATH = '/kaggle/working/TextEconomizer_WMT19_BESTPATH_20_V3.pth'
VAL_BEST_PATH = '/kaggle/working/TextEconomizer_WMT19_VAL_BEST_PATH_20_V3.pth'

In [ ]:
print("incorporating model checkpoint")

if os.path.exists(PATH):
    checkpoint = torch.load(PATH)
    model.load_state_dict(checkpoint['model_state_dict'])
    epoch = checkpoint['epoch']
    loss = checkpoint['loss']

print("incorporated model checkpoint")

In [ ]:
print("incorporating best model checkpoint")

if os.path.exists(BEST_PATH):
    checkpoint = torch.load(BEST_PATH)
    model.load_state_dict(checkpoint['model_state_dict'])
    epoch = checkpoint['epoch']
    loss = checkpoint['loss']

print("incorporated best model checkpoint")

In [ ]:
print("transfering the checkpoint")

P_PATH = '/kaggle/input/texteconomizer-20/pytorch/wmt19_v2/1/TextEconomizer_WMT19_20_V2.pth'

if os.path.exists(P_PATH):
    checkpoint = torch.load(P_PATH)
    model.load_state_dict(checkpoint['model_state_dict'])
    epoch = checkpoint['epoch']
    loss = checkpoint['loss']

epoch += 1

print("checkpoint transfered")

In [ ]:
### print("incorporating best validation checkpoint")

# if os.path.exists(VAL_BEST_PATH):
#     checkpoint = torch.load(VAL_BEST_PATH)
#     val_loss_checkpoint = checkpoint['val_loss']

# if epoch == 0:
#     best_val_loss = 10e9
# else:
#     best_val_loss = 5.348159131796463

best_val_loss = 1.898
# best_val_loss = 10e9

# print("incorporated best validation checkpoint")

In [ ]:
print("training has started")

best_epoch = 0

# scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=2, eta_min=5e-5)

for epoch in range(epoch, N_EPOCHS):

    start_time = timer()

    model.train()
    epoch_loss = 0

    optimizer.zero_grad()  # Initialize the gradients

    for i, (src, tgt) in enumerate(tqdm(train_dataloader)):
        src = src.to(DEVICE)
        tgt = tgt.to(DEVICE)

        tgt_input = tgt[:-1, :]

        src_mask, tgt_mask, src_padding_mask, tgt_padding_mask = create_mask(src, tgt_input)

        logits = model(src, tgt_input, src_mask, tgt_mask, src_padding_mask, tgt_padding_mask, src_padding_mask)

        tgt_out = tgt[1:, :]
        loss = loss_fn(logits.reshape(-1, logits.shape[-1]), tgt_out.reshape(-1))

        loss.backward()  # Backpropagate the loss

        optimizer.step()  # Update model weights
        optimizer.zero_grad()  # Reset gradients for the next batch

        epoch_loss += loss.item()  # Accumulate the true loss

    end_time = timer()

    epoch_loss = epoch_loss / len(train_dataloader)  # Average loss over the epoch

    val_loss = evaluate(model)

    print(f"Loss = {loss}")
    print(f"Epoch Loss = {epoch_loss}")

    print(f"Epoch: {epoch}, Train loss: {epoch_loss:.3f}, Val loss: {val_loss:.3f}, Epoch time = {(end_time - start_time):.3f}s")
    print()

    # ===========================================Model Saving=================================
    if epoch_loss < loss:
        loss = epoch_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'loss': loss,
        }, BEST_PATH)
        best_epoch = epoch
        print(f"{'-'*50}\nModel Saved at {BEST_PATH}\n{'-'*50}\n")
    else:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'loss': loss,
        }, PATH)
        print(f"{'-'*50}\nModel Saved at {PATH}\n{'-'*50}\n")

    # ========================================Best Validation Model Saving===================
    if best_val_loss > val_loss:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'loss': loss,
            'val_loss': val_loss,
        }, VAL_BEST_PATH)
        best_val_loss = val_loss
        print(f"{'-'*50}\nModel Saved at {VAL_BEST_PATH}\n{'-'*50}\n")


    # scheduler.step()

    # for param_group in optimizer.param_groups:
    #     print(f"Epoch {epoch + 1} Learning Rate: {param_group['lr']}")

    print(f"{'-'*50}\nThe Best Epoch Number: {best_epoch}\n{'-'*50}\n")

print(f"{'='*25}training ended{'='*25}")

In [ ]:
print("incorporating latest model checkpoint")

PATH = '/kaggle/input/nuggets-20-checkpoint/pytorch/evaluation/1/transformer_PwC_VAL_BEST_PATH_20_V7.pth'

if os.path.exists(PATH):
    checkpoint = torch.load(PATH)
    model.load_state_dict(checkpoint['model_state_dict'])
    epoch = checkpoint['epoch']
    loss = checkpoint['loss']

print("incorporated latest model checkpoint")

In [ ]:
def evaluate(model, val_dataloader, loss_fn, DEVICE):
    model.eval()
    losses = 0

    for src, tgt in val_dataloader:
        src = src.to(DEVICE)
        tgt = tgt.to(DEVICE)

        tgt_input = tgt[:-1, :]

        src_mask, tgt_mask, src_padding_mask, tgt_padding_mask = create_mask(src, tgt_input)

        logits = model(src, tgt_input, src_mask, tgt_mask,src_padding_mask, tgt_padding_mask, src_padding_mask)

        tgt_out = tgt[1:, :]
        loss = loss_fn(logits.reshape(-1, logits.shape[-1]), tgt_out.reshape(-1))
        losses += loss.item()

    return losses / len(list(val_dataloader))

In [ ]:
test_loss = evaluate(model, val_dataloader, loss_fn, DEVICE)

print(f"| Test Loss: {test_loss:.3f} | Test PPL: {np.exp(test_loss):7.3f} |")

# **Run Inference**

In [ ]:
# function to generate output sequence using greedy algorithm
def greedy_decode(model, src, src_mask, max_len, start_symbol):
    src = src.to(DEVICE)
    src_mask = src_mask.to(DEVICE)

    memory = model.encode(src, src_mask)
    ys = torch.ones(1, 1).fill_(start_symbol).type(torch.long).to(DEVICE)
    for i in range(max_len-1):
        memory = memory.to(DEVICE)
        tgt_mask = (generate_square_subsequent_mask(ys.size(0))
                    .type(torch.bool)).to(DEVICE)
        out = model.decode(ys, memory, tgt_mask)
        out = out.transpose(0, 1)
        prob = model.generator(out[:, -1])
        _, next_word = torch.max(prob, dim=1)
        next_word = next_word.item()

        ys = torch.cat([ys,
                        torch.ones(1, 1).type_as(src.data).fill_(next_word)], dim=0)
        if next_word == EOS_IDX:
            break
    return ys


# actual function to translate input sentence into target language
def translate(model: torch.nn.Module, src_sentence: str):
    model.eval()
    src = text_transform[SRC_LANGUAGE](src_sentence).view(-1, 1)
    num_tokens = src.shape[0]
    src_mask = (torch.zeros(num_tokens, num_tokens)).type(torch.bool)
    tgt_tokens = greedy_decode(
        model,  src, src_mask, max_len=num_tokens + 5, start_symbol=BOS_IDX).flatten()
    return " ".join(vocab_transform[TGT_LANGUAGE].lookup_tokens(list(tgt_tokens.cpu().numpy()))).replace("<bos>", "").replace("<eos>", "")

In [ ]:
# They are currently working on a new project, but they will be attending a conference next week, and after that, they could have a well-deserved vacation.
translation = translate(model, "They currently working on a new project, but they attending a conference next week, and after that, they a well-deserved vacation.")
print(translation)

In [ ]:
# She had been studying for hours, and by the time you arrived, she will have completed her assignment, and then she might have taken a short break.
translation = translate(model, "She studying for hours, and by the time you arrived, she completed her assignment, and then she taken a short break.")
print(translation)

In [ ]:
#Example 1:
# She had been studying for hours, and by the time you arrived, she will have completed her assignment, and then she might have taken a short break.
translation = translate(model, "In some cases the number is 120,000,130,000")
print(translation)

# **Evaluation**

In [ ]:
!pip install sacrebleu
!pip install bert_score evaluate
!pip install torchmetrics
!pip install rouge_score

In [ ]:
import datasets
from evaluate import load
import nltk
from nltk.translate.bleu_score import sentence_bleu
import sacrebleu
import evaluate


sacrebleu = evaluate.load("sacrebleu")
bertscore = load("bertscore")
bleu = evaluate.load("bleu")
rouge = evaluate.load('rouge')
meteor = evaluate.load('meteor')

In [ ]:
def calculate_wer(reference, candidate):
    reference = reference.split()
    candidate = candidate.split()

    # Create a matrix to store distances
    distance_matrix = np.zeros((len(reference) + 1) * (len(candidate) + 1), dtype=int).reshape((len(reference) + 1, len(candidate) + 1))

    for i in range(len(reference) + 1):
        for j in range(len(candidate) + 1):
            if i == 0:
                distance_matrix[i][j] = j
            elif j == 0:
                distance_matrix[i][j] = i
            elif reference[i - 1] == candidate[j - 1]:
                distance_matrix[i][j] = distance_matrix[i - 1][j - 1]
            else:
                distance_matrix[i][j] = 1 + min(distance_matrix[i][j - 1],      # Insert
                                               distance_matrix[i - 1][j],      # Remove
                                               distance_matrix[i - 1][j - 1])  # Replace

    return distance_matrix[len(reference)][len(candidate)]

In [ ]:
import re

def autoregressive_decoding(sentence):

    if isinstance(sentence, list):
        sentence = ' '.join(sentence)
    
    # Remove subword markers "##" by merging with the preceding word
    sentence = sentence.replace(" ##", "")

    # Remove spaces before punctuation (comma, period, etc.)
    sentence = re.sub(r'\s+([.,!?;:])', r'\1', sentence)

    # Remove spaces around single and double quotes
    sentence = re.sub(r"\s*(['\"])\s*", r'\1', sentence)

    # Remove spaces around apostrophes in contractions
    sentence = re.sub(r"\s*'\s*(\w+)", r"'\1", sentence)

    # Replace multiple spaces with a single space
    sentence = re.sub(r'\s+', ' ', sentence).strip()

    # Remove spaces after opening and before closing parentheses/brackets
    sentence = re.sub(r'\s*\(\s*', '(', sentence)
    sentence = re.sub(r'\s*\)\s*', ')', sentence)

    # Ensure proper ellipsis formatting (already in the original code)
    sentence = re.sub(r'\s*\.\s*\.\s*\.\s*', '...', sentence)

    # Remove spaces around hyphens or dashes
    sentence = re.sub(r'\s*-\s*', '-', sentence)

    # Place punctuation inside quotation marks (handle most common cases)
    sentence = re.sub(r'([.,!?;:])\s*([\'"])', r'\2\1', sentence)

    # Remove spaces around "@" for email addresses and Twitter handles
    sentence = re.sub(r'\s*@\s*', '@', sentence)

    # Remove spaces around slashes (e.g., in URLs or subreddit names)
    sentence = re.sub(r'\s*/\s*', '/', sentence)

    # Remove spaces around currency symbols and numbers
    sentence = re.sub(r'\s*([$€£])\s*([0-9]+)', r' \1\2', sentence)
    sentence = re.sub(r'([0-9]+)\s*([,])\s*([0-9]+)', r'\1\2\3', sentence)
    sentence = re.sub(r'([0-9]+)\s*([%])', r'\1\2', sentence)

    # Remove spaces within URLs
    sentence = re.sub(r'\s*(http://|https://|www\.)\s*', r'\1', sentence)

    # Remove spaces in dates and times
    sentence = re.sub(r'\s*/\s*', '/', sentence)  # For dates with slashes
    sentence = re.sub(r'\s*:\s*', ':', sentence)  # For times or other similar cases

    # Remove spaces around mathematical operators
    sentence = re.sub(r'\s*([+\-*/=])\s*', r'\1', sentence)

    # Remove spaces around file extensions
    sentence = re.sub(r'\s+(\.[a-zA-Z0-9]+)', r'\1', sentence)

    # Remove spaces around code-like markers (assuming `code` is delimited by backticks)
    sentence = re.sub(r'\s*(```)\s*', r'\1', sentence)

    # --- New logic for removing spaces after dots ---

    # 1. Remove spaces after a dot when followed by a number (for decimals, currency, dates)
    sentence = re.sub(r'([0-9])\.\s+([0-9])', r'\1.\2', sentence)

    # 2. Remove spaces after a dot when followed by a letter (for abbreviations like U.S.A)
    sentence = re.sub(r'([A-Za-z])\.\s+([A-Za-z])', r'\1.\2', sentence)

    # 3. Remove spaces after a dot when followed by punctuation (e.g., ". , ;")
    sentence = re.sub(r'\.\s+([.,!?;:])', r'.\1', sentence)

    # Ensure there's a space before an opening parenthesis if preceded by a word
    sentence = re.sub(r'(\w)\(', r'\1 (', sentence)

    # Ensure there's a space after a closing parenthesis if followed by a word
    sentence = re.sub(r'\)(\w)', r') \1', sentence)

    # Ensure there's a space after a closing parenthesis if followed by any symbol (%, $, etc.)
    sentence = re.sub(r'\)([^\w\s])', r') \1', sentence)

    # Remove spaces between closing parenthesis and punctuation
    sentence = re.sub(r'\)\s+([,.;!?])', r')\1', sentence)

    # 4. Handle cases where a dot is followed by multiple dots but not an ellipsis (e.g., "U.S. ...")
    sentence = re.sub(r'\.\s+\.', r'..', sentence)

    # Add spaces around single or double quotes for 'protected' words
    sentence = re.sub(r"(\S)(['\"])(\w+)(['\"])(\S)", r"\1 \2\3\4 \5", sentence)

    return sentence

In [ ]:
import re
from tqdm import tqdm

print("evaluation begins!")

total_bleu_score = 0
total_sacre_bleu_score = 0
total_wer_score = 0
total_precision = 0
total_recall = 0
total_f1 = 0

# Initialize variables for Rouge scores
total_rouge1_score = 0
total_rouge2_score = 0
total_rougeL_score = 0

# Initialize variable for METEOR score
total_meteor_score = 0

# Create lists to store individual scores for each sentence
bleu_scores = []
sacre_bleu_scores = []
wer_scores = []
precision_scores = []
recall_scores = []
f1_scores = []

# Create lists to store Rouge scores for each sentence
rouge1_scores = []
rouge2_scores = []
rougeL_scores = []

# Create list to store METEOR scores for each sentence
meteor_scores = []

# sacrebleu_hf = load_metric("sacrebleu")
# bertscore = load("bertscore")

for (source_sentence, target_sentence) in tqdm(eval_dataset):

    transformer_output = translate(model, source_sentence)
    output_text = autoregressive_decoding(transformer_output)
    target_sentence = target_sentence.lower()
    # print(f"\nDAE output: {output_text}\ntarget sentence: {target_sentence}\n")

    # Calculate BLEU score using SacreBLEU
    temp_bleu_score = bleu.compute(predictions=[output_text], references=[[target_sentence]], max_order=2)
    bleu_score = temp_bleu_score["bleu"]

    # Calculate Rouge scores
    temp_rouge_score = rouge.compute(predictions=[output_text], references=[target_sentence])
    rouge1_score = temp_rouge_score["rouge1"]
    rouge2_score = temp_rouge_score["rouge2"]
    rougeL_score = temp_rouge_score["rougeL"]

    # Calculate METEOR score
    temp_meteor_score = meteor.compute(predictions=[output_text], references=[target_sentence])
    meteor_score = temp_meteor_score["meteor"]

    # Calculate SacreBLEU and WER scores
    sacre_bleu_score = sacrebleu.compute(predictions=[output_text], references=[[target_sentence]])
    sacre_bleu_score = sacre_bleu_score["score"]
    wer_score = calculate_wer(target_sentence, output_text)

    # Calculate BERTScore metrics
    results = bertscore.compute(predictions=[output_text], references=[target_sentence], lang="en")

    precision_score = results["precision"]
    recall_score = results["recall"]
    f1_score = results["f1"]

    if isinstance(precision_score, list):
        precision_score = sum(precision_score) / len(precision_score)
    if isinstance(recall_score, list):
        recall_score = sum(recall_score) / len(recall_score)
    if isinstance(f1_score, list):
        f1_score = sum(f1_score) / len(f1_score)

    # Accumulate scores
    total_bleu_score += bleu_score

    total_rouge1_score += rouge1_score
    total_rouge2_score += rouge2_score
    total_rougeL_score += rougeL_score

    total_meteor_score += meteor_score

    total_sacre_bleu_score += sacre_bleu_score
    total_wer_score += wer_score

    total_precision += precision_score
    total_recall += recall_score
    total_f1 += f1_score

    # Store individual scores for each sentence
    bleu_scores.append(bleu_score)
    rouge1_scores.append(rouge1_score)
    rouge2_scores.append(rouge2_score)
    rougeL_scores.append(rougeL_score)

    meteor_scores.append(meteor_score)

    sacre_bleu_scores.append(sacre_bleu_score)
    wer_scores.append(wer_score)

    precision_scores.append(precision_score)
    recall_scores.append(recall_score)
    f1_scores.append(f1_score)


# Calculate average scores
avg_bleu_score = total_bleu_score / len(eval_dataset)
avg_sacre_bleu_score = total_sacre_bleu_score / len(eval_dataset)
avg_wer_score = total_wer_score / len(eval_dataset)

avg_rouge1_score = total_rouge1_score / len(eval_dataset)
avg_rouge2_score = total_rouge2_score / len(eval_dataset)
avg_rougeL_score = total_rougeL_score / len(eval_dataset)

avg_meteor_score = total_meteor_score / len(eval_dataset)

avg_PR_score = total_precision / len(eval_dataset)
avg_RE_score = total_recall / len(eval_dataset)
avg_F1_score = total_f1 / len(eval_dataset)

# Print average scores
print()
print(f"Average BLEU Score: {(avg_bleu_score * 100):.4f}")
print(f"Average SacreBLEU Score: {avg_sacre_bleu_score:.4f}")
print(f"Average Word Error Rate (WER): {avg_wer_score}")

print(f"Average Rouge-1 Score: {avg_rouge1_score}")
print(f"Average Rouge-2 Score: {avg_rouge2_score}")
print(f"Average Rouge-L Score: {avg_rougeL_score}")

print(f"Average METEOR Score: {avg_meteor_score:.4f}")

print(f"Average Precision: {avg_PR_score}")
print(f"Average Recall: {avg_RE_score}")
print(f"Average F1-Score: {avg_F1_score}")

print("evaluation ends!")

In [ ]:
def evaluate(model, val_dataloader, loss_fn, DEVICE):
    model.eval()
    losses = 0

    for src, tgt in val_dataloader:
        src = src.to(DEVICE)
        tgt = tgt.to(DEVICE)

        tgt_input = tgt[:-1, :]

        src_mask, tgt_mask, src_padding_mask, tgt_padding_mask = create_mask(src, tgt_input)

        logits = model(src, tgt_input, src_mask, tgt_mask,src_padding_mask, tgt_padding_mask, src_padding_mask)

        tgt_out = tgt[1:, :]
        loss = loss_fn(logits.reshape(-1, logits.shape[-1]), tgt_out.reshape(-1))
        losses += loss.item()

    return losses / len(list(val_dataloader))

In [ ]:
test_loss = evaluate(model, val_dataloader, loss_fn, DEVICE)

print(f"| Test Loss: {test_loss:.3f} | Test PPL: {np.exp(test_loss):7.3f} |")

In [ ]:
print(translate(model, 'She studying for hours, and by the time you arrived, she completed her assignment, and then she taken a short break.'))